# Capstone #3: Production-Pattern Customer Service Agent

*Notebook 24*

Put it all together.

Combine sessions, guardrails, and human-in-the-loop into a customer service pipeline built on production patterns.

Demo stores stand in for real databases and auth.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunConfig,
    Runner,
    SQLiteSession,
    function_tool,
    input_guardrail,
)

# Notebook-specific imports
import json

MODEL = "gpt-5-mini"

print(f"✅ Setup complete: Using {MODEL}")

---

## 🏗️ System Architecture

This capstone combines sessions (Lesson 17), guardrails (Lesson 20), and human-in-the-loop approval (Lesson 22) into a single production-pattern pipeline:

**Pipeline:**
```
Customer message
        ↓
[Input Guardrail]: blocks requests classified as off-topic
        ↓
Support Agent (with SQLiteSession)
    ├── lookup_order          → auto-executes
    ├── search_faq            → auto-executes
    └── process_refund       → pauses for approval
        ↓
[Approval flow] → auto or manual based on the trusted order price
        ↓
Response (saved to session)
```
Runs are traced by default for later inspection in the OpenAI dashboard.

---

## 🛠️ Phase 1: Define the Tools

In [ ]:
# Simulated order database (a real system would use your orders service)
ORDERS = {
    "ORD-001": {"item": "Wireless Headphones", "price": 149.99, "status": "delivered", "date": "2025-12-10", "customer_id": "CUST-1001"},
    "ORD-002": {"item": "Phone Case", "price": 19.99, "status": "delivered", "date": "2025-12-12", "customer_id": "CUST-1001"},
    "ORD-003": {"item": "Laptop Stand", "price": 49.99, "status": "in transit", "date": "2025-12-15", "customer_id": "CUST-1001"},
    "ORD-004": {"item": "USB-C Cable", "price": 12.99, "status": "delivered", "date": "2025-12-08", "customer_id": "CUST-2002"}
}

# Refund ledger keyed by (customer_id, order_id): proves execution and blocks duplicates
PROCESSED_REFUNDS = {}

# Evidence log: every support-tool execution appends its name here
TOOL_CALL_LOG = []

# Simulated FAQ database
FAQS = [
    {"q": "return policy", "a": "We accept returns on eligible delivered orders. Items must be in original condition. Contact us to confirm eligibility."},
    {"q": "shipping time", "a": "Standard shipping takes 5-7 business days. Express shipping takes 2 business days."},
    {"q": "warranty", "a": "All electronics come with a 1-year manufacturer warranty. Contact us to initiate a claim."},
    {"q": "payment", "a": "We accept Visa, Mastercard, Amex, and PayPal. Payment is charged at time of order."}
]


# Trusted application identity, owned by the application, not the model.
# DEMO_CUSTOMER_ID is not exposed as a model-supplied tool argument, so the model
# cannot choose or replace the identity used by validation.
# Single-customer demo: a real application binds authenticated identity to each
# request through its authentication layer, outside model-controlled input, not a shared global.
DEMO_CUSTOMER_ID = "CUST-1001"


@function_tool
def lookup_order(order_id: str) -> str:
    """Look up the details and status of one of the customer's own orders."""
    TOOL_CALL_LOG.append("lookup_order")
    order = ORDERS.get(order_id.upper())
    if not order or order["customer_id"] != DEMO_CUSTOMER_ID:
        return f"Order {order_id} not found for your account. Please check the order ID."
    return (
        f"Order {order_id}: {order['item']}, ${order['price']:.2f}\n"
        f"Status: {order['status']} | Date: {order['date']}"
    )


@function_tool
def search_faq(query: str) -> str:
    """Search the FAQ database for answers to common customer questions."""
    TOOL_CALL_LOG.append("search_faq")
    query_lower = query.lower()
    for faq in FAQS:
        if faq["q"] in query_lower or any(word in query_lower for word in faq["q"].split()):
            return faq["a"]
    return "I couldn't find a specific FAQ for that question. Please contact our support team directly."


# needs_approval=True pauses the run and populates result.interruptions.
# The read-only tools above run without approval because needs_approval
# defaults to False: the SDK does not detect side effects, you declare them.
# The refund amount comes from the order record, never from the model.
@function_tool(needs_approval=True)
def process_refund(order_id: str, reason: str) -> str:
    """Refund one of the customer's delivered orders at its recorded price. Requires approval."""
    TOOL_CALL_LOG.append("process_refund")
    key = (DEMO_CUSTOMER_ID, order_id.upper())
    order = ORDERS.get(order_id.upper())
    if not order or order["customer_id"] != DEMO_CUSTOMER_ID:
        return f"Refund declined: order {order_id} not found for your account."
    if order["status"] != "delivered":
        return f"Refund declined: order {order_id} has not been delivered (status: {order['status']})."
    if key in PROCESSED_REFUNDS:
        return f"Refund declined: order {order_id} was already refunded."
    amount = order["price"]  # trusted price from the order record
    PROCESSED_REFUNDS[key] = amount
    return f"✅ Refund of ${amount:.2f} processed for order {order_id}. Reason: {reason}"


# --------------------------------------------------------------
print("✅ Tools defined")
print("   lookup_order    no approval needed")
print("   search_faq      no approval needed")
print("   process_refund  requires approval")

⚠️ **Security note:** The model still chooses `order_id` and `reason`. The pipeline validates the order against trusted data before approving; the reason is free text kept for the audit record.

The refund amount comes from the order record, and the customer identity comes from `DEMO_CUSTOMER_ID`. It is not exposed as a model-supplied tool argument, so the model cannot choose or replace the identity used by validation. This is a single-customer demo: a real application binds authenticated identity to each request through its authentication layer, outside model-controlled input, and would not use a shared global.

---

## 🛡️ Phase 2: Block Off-Topic Requests

A guardrail agent screens the message before the support agent runs. When the classifier trips, the request is rejected up front.

We pass `run_in_parallel=False` to make the guardrail blocking: decorated guardrails otherwise run concurrently with the agent.

A keyword check would be brittle. A classifier agent can judge topics beyond fixed keywords, though its verdict is still model-generated.

In [ ]:
class SupportTopicCheck(BaseModel):
    is_support_related: bool
    reasoning: str


topic_checker_instructions = (
    "Determine if the message is related to customer support for an online store.\n"
    "Support topics include: orders, refunds, returns, shipping, products, payments, warranties, and account issues.\n"
    "Return is_support_related=True only if it's clearly a customer support request."
)

topic_checker = Agent(
    name="TopicChecker",
    instructions=topic_checker_instructions,
    model=MODEL,
    output_type=SupportTopicCheck
)


# run_in_parallel=False makes this guardrail blocking: it finishes before
# the support agent starts, so a tripwire means no support-agent turn ran.
@input_guardrail(run_in_parallel=False)
async def support_topic_guardrail(
    ctx, agent: Agent, input
) -> GuardrailFunctionOutput:
    """Block requests unrelated to customer support."""

    result = await Runner.run(topic_checker, input)

    check = result.final_output
    return GuardrailFunctionOutput(
        tripwire_triggered=not check.is_support_related,
        output_info=check.reasoning
    )


# --------------------------------------------------------------
print("✅ Support topic guardrail ready")

---

⚠️ **Security note:** Customer messages are untrusted input. The input guardrail reduces the attack surface but cannot block prompt injection inside support-related messages.

Even a blocked message is saved to the session you pass: the SDK stores the input before the guardrail decides.

Treat tool outputs and retrieved content as data, not instructions. (See Lesson 21)

---

## 🤖 Phase 3: Build the Agent

In [ ]:
support_instructions = (
    "You are a helpful customer support agent for an online electronics store.\n\n"
    "You can:\n"
    "- Look up order details and status with lookup_order\n"
    "- Answer questions about policies with search_faq\n"
    "- Process refunds with process_refund (for eligible delivered orders)\n\n"
    "After using a tool, answer only from information in its result.\n"
    "Do not add policy details, timelines, or follow-up services that the result does not support.\n"
    "You cannot arrange replacements, repairs, return labels, emails, or receipts.\n\n"
    "Always be polite, empathetic, and solution-focused.\n"
    "When the customer explicitly asks you to process a refund, look up the order first, then call process_refund."
)

support_agent = Agent(
    name="CustomerSupport",
    instructions=support_instructions,
    model=MODEL,
    tools=[lookup_order, search_faq, process_refund],
    input_guardrails=[support_topic_guardrail]
)


# --------------------------------------------------------------
print("✅ Customer support agent ready")
print("   Tools: lookup_order, search_faq, process_refund")
print("   Guardrail: support topic filter")
print("   Session: will be passed at runtime")

## 🚀 Phase 4: The Full Pipeline

The orchestration layer handles sessions, guardrails, and the approval flow.

Focus on one idea here: `handle_customer_message()` is a plain Python wrapper around `Runner.run()` where you add app-specific rules: validation, approval thresholds, logging, and fallback behavior.

**Fail closed:** unknown, unowned, undelivered, or already-refunded orders reject before any approval.

**Threshold routing:** with `auto_approve=True`, validated refunds of USD 100 or less auto-approve using the trusted order price. Higher amounts pause for manual review.

In [ ]:
AUTO_APPROVE_THRESHOLD = 100.0


def validate_refund_request(order_id):
    """Fail closed: pass only a real, owned, delivered, not-yet-refunded order."""
    order = ORDERS.get(order_id.upper()) if isinstance(order_id, str) else None
    if not order or order["customer_id"] != DEMO_CUSTOMER_ID:
        return None, "order not found for this customer"
    if order["status"] != "delivered":
        return None, f"order not delivered (status: {order['status']})"
    if (DEMO_CUSTOMER_ID, order_id.upper()) in PROCESSED_REFUNDS:
        return None, "order already refunded"
    return order, None


async def handle_customer_message(message, session, auto_approve=True, agent=None):
    """Handle a customer message with guardrail, session, and approval routing."""

    run_agent = agent or support_agent
    run_config = RunConfig(workflow_name="Customer support pipeline")

    # -------------------------------------------------------
    # Component 1: Guardrail check + initial run
    # -------------------------------------------------------
    try:
        result = await Runner.run(
            run_agent, input=message, session=session,
            run_config=run_config,
        )

    except InputGuardrailTripwireTriggered:
        return "I'm sorry, I can only help with customer support questions about orders, returns, and products."

    # -------------------------------------------------------
    # Component 2: Human-in-the-loop approval flow
    # -------------------------------------------------------
    # A tool marked needs_approval=True pauses the run and populates
    # result.interruptions. Validate against trusted order data FIRST,
    # then route by the trusted price: approval checks permission,
    # validation checks correctness.
    while result.interruptions:
        state = result.to_state()

        for interruption in state.get_interruptions():
            if interruption.raw_item.name != process_refund.name:
                state.reject(interruption, rejection_message="No approval policy for this tool.")
                continue

            try:
                args = json.loads(interruption.raw_item.arguments)
            except (json.JSONDecodeError, TypeError):
                args = None
            order_id = args.get("order_id") if isinstance(args, dict) else None
            reason = args.get("reason") if isinstance(args, dict) else None
            if not isinstance(order_id, str) or not isinstance(reason, str) or not reason.strip():
                state.reject(interruption, rejection_message="Malformed tool arguments.")
                continue

            order, problem = validate_refund_request(order_id)
            if problem:
                print(f"   🚫 Rejected: {problem}")
                state.reject(interruption, rejection_message=f"Refund declined: {problem}.")
                continue

            amount = order["price"]  # trusted price, not a model-supplied argument

            if auto_approve and amount <= AUTO_APPROVE_THRESHOLD:
                print(f"   ✅ Auto-approved: ${amount:.2f} refund")
                state.approve(interruption)
            else:
                print(f"   ⚠️  Manual review: ${amount:.2f} refund for {order_id}")
                print(f"      Reason: {reason}")
                decision = input("      Approve? (yes/no): ").strip().lower()
                if decision in ["yes", "y"]:
                    print("      ✅ Approved by human")
                    state.approve(interruption)
                else:
                    print("      🚫 Rejected by human")
                    state.reject(interruption, rejection_message="Refund requires additional verification.")

        # The state carries the pending run; pass the session so the
        # resumed turn's tool output and final response are saved to it.
        result = await Runner.run(run_agent, state, session=session, run_config=run_config)

    return result.final_output


# --------------------------------------------------------------
print("✅ handle_customer_message() ready")
print(f"   Auto-approve threshold: ${AUTO_APPROVE_THRESHOLD}")

### 🔭 Tracing

Runs are traced by default, and the handler names them with `workflow_name="Customer support pipeline"`.

Open `https://platform.openai.com/traces`, find that workflow, and look for:

- The `support_topic_guardrail` span: the guardrail name and whether it tripped

- The `process_refund` function span: appears once an approved run resumes and the tool executes

- The resumed run: reattaches to the same trace as the paused run

- Token counts across turns: watch how multi-turn sessions grow

### 💡 Key Takeaway

`lookup_order` and `search_faq` run without approval because `needs_approval` defaults to `False`.

`process_refund` declares `needs_approval=True` because a refund is high-impact.

The SDK does not detect side effects: you declare which tools need approval.

---

⚠️ Turn 2 is designed to pause if the agent calls `process_refund`. ORD-001's USD 149.99 exceeds the USD 100 auto-approve threshold. When it pauses, you'll see an `input()` prompt: type `yes` to approve.

Watch turn 2. The customer doesn't repeat the order ID. The session is what lets the agent carry ORD-001 forward, so customers don't restate context every turn.

The grounding instructions guide model output. Validation and approval gates enforce the high-impact behavior.

In [ ]:
# One session per conversation thread. SQLiteSession with no file path is
# in-memory demo state: production would pass a database file path.
customer_session = SQLiteSession("customer_demo_001")

conversation = [
    "Hi, my order is ORD-001. I need help with my headphones.",
    "They stopped working after 3 days. Please process a full refund for the headphones order I mentioned.",
    "What's your return policy?",
]

# Reset demo state so re-running the notebook doesn't reject its own refund as a duplicate
PROCESSED_REFUNDS.clear()
TOOL_CALL_LOG.clear()

print("🎬 CUSTOMER SERVICE DEMO")
print("=" * 60)
print(f"PROCESSED_REFUNDS before: {PROCESSED_REFUNDS}")

for turn_number, message in enumerate(conversation, 1):
    print(f"\n👤 Customer (Turn {turn_number}): {message}")
    response = await handle_customer_message(message, customer_session)
    print(f"🤖 Agent: {response}")
    print("-" * 60)

print(f"\nPROCESSED_REFUNDS after: {PROCESSED_REFUNDS}")
print(f"TOOL_CALL_LOG: {TOOL_CALL_LOG}")

#### Test: Off-Topic Request (Guardrail)

In [ ]:
print("\n🛡️  GUARDRAIL TEST")
print("=" * 60)
off_topic = "Can you help me write a Python script?"
print(f"👤 Customer: {off_topic}")

# Separate session so the blocked message doesn't pollute the customer's thread
guardrail_test_session = SQLiteSession("guardrail_test")

tools_before = len(TOOL_CALL_LOG)
response = await handle_customer_message(off_topic, guardrail_test_session)

print(f"🤖 Agent: {response}")
print(f"🔧 Support tools called during the blocked run: {TOOL_CALL_LOG[tools_before:]}")

saved = await guardrail_test_session.get_items()
print(f"\n📌 The blocked message was still saved to its session ({len(saved)} item):")
print("   the SDK stores the input before the guardrail decides.")

#### Cleanup

In [ ]:
await customer_session.clear_session()
customer_session.close()

await guardrail_test_session.clear_session()
guardrail_test_session.close()

print("✅ Sessions cleared and closed")

---

A useful extension pattern is to parameterize your orchestration function with `agent`, so you can swap in upgraded versions without rewriting the approval flow.

### Exercise 1: Extend the Customer Service Agent

*Covers: agent extension (adding tools, guardrails, and evaluation)*

Extend the agent with a new tool, a stricter guardrail, and an evaluation pass.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Extend the Customer Service Agent
# --------------------------------------------------------------
# Objective: Extend the customer service agent with new capabilities.
#
# Phases 1-4 (tools, guardrail, agent, pipeline) are provided above.
# Your job: implement the two extensions below.

# -------------------------------------------------------
# Extension 1: check_shipping_status tool
# -------------------------------------------------------
# TODO 1: Create a @function_tool called check_shipping_status
#            - Parameters: order_id (str)
#            - Return data only when the order belongs to DEMO_CUSTOMER_ID
#            - Look up the order in ORDERS and return a shipping status message
#            - No approval needed

# -------------------------------------------------------
# Extension 2: Update the agent
# -------------------------------------------------------
# TODO 2: Create extended_support_agent, copy support_agent
#            but add check_shipping_status to the tools list
#            and update the instructions to mention it

# -------------------------------------------------------
# Extension 2b: Stricter guardrail
# -------------------------------------------------------
# TODO 2b: Update topic_checker_instructions or create an enhanced guardrail
#            that also rejects competitor comparison requests
#            (e.g. "How does your return policy compare to Amazon's?")
#            Wire the updated guardrail into extended_support_agent

# -------------------------------------------------------
# Extension 3: Test it
# -------------------------------------------------------
# TODO 3: Create a new SQLiteSession (identity is DEMO_CUSTOMER_ID)
#            Run handle_customer_message() passing extended_support_agent
#            via its agent parameter
#            Test with: "Where is my order ORD-003?"
#            Print the response

# -------------------------------------------------------
# Extension 4: Evaluate the agent
# -------------------------------------------------------
# TODO 4: Build a small golden test set: 3 to 5 test cases covering:
#            - A valid order lookup
#            - A valid refund request
#            - An off-topic request that should be blocked
#           Run each case through handle_customer_message() so refunds
#           can clear approval. Snapshot len(TOOL_CALL_LOG) before each
#           case so you judge only that case's tool calls, and use a
#           validated low-value order (ORD-002) for the refund case.
#           Capture the evidence as you go:
#           TOOL_CALL_LOG for tools called, PROCESSED_REFUNDS for side
#           effects, and the returned reply for the blocked case.
#           Check tool use, guardrail behavior, and side effects in code.
#           Use the judge agent pattern from Lesson 07 only to score the
#           helpfulness and accuracy of each captured response.
#           Clear and close your exercise session when done.

print("💡 Implement the extensions above, then test them!")

---

## 🎯 Key Takeaways

**Production features work together:**

- Sessions preserve conversation context across turns

- A blocking input guardrail (`run_in_parallel=False`) screens requests before the agent runs

- Human-in-the-loop approval gates high-impact actions

<br>
<br>

**Fail closed before approving:**

- Validate against trusted data first: ownership, delivered status, recorded price

- Derive amounts from the order record, never from model-supplied arguments

- Record executed refunds and reject duplicates before they reach approval

<br>
<br>

**Auto-approve small actions, escalate large ones:**

- With `auto_approve=True`, validated small refunds auto-approve against the trusted price, larger ones go to a human

- The threshold must compare trusted data, not tool arguments

<br>
<br>

**Know the boundaries:**

- The topic guardrail reduces the attack surface but does not block injection inside support-related messages

- Even a blocked message is saved to the session you pass

- Tools run without approval because `needs_approval` defaults to `False`, not because the SDK detects side effects

- These are production patterns on demo stores: real systems add auth, durable databases, and audit trails

---

## 📍 Next Step

**Notebook 25: MCP Fundamentals**  

Connect your agents to the growing ecosystem of MCP servers for filesystem access, web browsing, and more.

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-24-capstone-3--customer-service-agent)

---